# A1 — Fractal Dimension & Multifractal Spectrum

**Goal:** Build intuition for what fractal analysis computes on painting images, verify the mathematics numerically, and test whether Manet and a contemporary are visually separable.

**Methods covered:**
1. Box-counting fractal dimension D
2. Sliding Window Fractal Dimension (SWFD) → spatial D-map
3. Lacunarity λ(ε)
4. Generalised dimension D(q)
5. Singularity spectrum f(α) via Legendre transform

**Key paper:** Petrić et al. (2025), *Symmetry* 17(5):703 — 96.6% accuracy, direct RGB support

---
⚠️ **Important warning (Jones-Smith & Mathur, 2006, *Nature*):** A single global D value is insufficient for authentication — trivial sketches can match Pollock's D. Always use local SWFD maps + multifractal Δα + ensemble.

In [ ]:
import sys
sys.path.insert(0, '../..')   # make repo root importable

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from notebooks.research.utils import load_image, to_gray, show_pair, heatmap_overlay

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 0. Configuration — set your image paths here

In [ ]:
# ── Set these paths to your client-provided images ───────────────────────────
PATH_MANET       = Path('../../data/manet/manet_example.tif')
PATH_CONTEMPORARY = Path('../../data/contemporary/contemporary_example.tif')
# ─────────────────────────────────────────────────────────────────────────────

# Load images — downsampled to 1024px long edge for speed
img_m = load_image(PATH_MANET)
img_c = load_image(PATH_CONTEMPORARY)

gray_m = to_gray(img_m)
gray_c = to_gray(img_c)

show_pair(img_m, img_c)
print(f'Manet:       {img_m.shape}  ({img_m.shape[1]}×{img_m.shape[0]} px)')
print(f'Contemporary:{img_c.shape}  ({img_c.shape[1]}×{img_c.shape[0]} px)')

---
## 1. Box-Counting Fractal Dimension

### Mathematics

For a fractal set, the number of boxes of side length $\varepsilon$ needed to cover the set scales as:
$$N(\varepsilon) \propto \varepsilon^{-D}$$

Taking logarithms:
$$\log N(\varepsilon) = -D \cdot \log(\varepsilon) + C$$

So $D$ is the **negative slope** of $\log N$ vs $\log \varepsilon$.

For an image we use pixel intensities as a probability measure:
- Divide the image into non-overlapping boxes of side $\varepsilon$
- Count boxes with total intensity above a threshold (or use all boxes weighted by mass)
- Fit the log-log regression

**Stability requirement:** Box sizes must span ≥ 2 orders of magnitude. Use a geometric sequence with factor 1.5.

In [ ]:
def box_count(gray: np.ndarray, box_sizes: np.ndarray) -> np.ndarray:
    """
    Count non-empty boxes at each scale.
    A box is 'non-empty' if its mean intensity exceeds the global mean.
    """
    threshold = gray.mean()
    counts = []
    H, W = gray.shape
    for s in box_sizes:
        s = int(s)
        # Trim image to be divisible by s
        h_trim = (H // s) * s
        w_trim = (W // s) * s
        blocks = gray[:h_trim, :w_trim].reshape(H // s, s, W // s, s)
        block_means = blocks.mean(axis=(1, 3))
        counts.append((block_means > threshold).sum())
    return np.array(counts, dtype=float)


def fractal_dimension(gray: np.ndarray, min_box: int = 2, factor: float = 1.5) -> tuple:
    """
    Compute box-counting fractal dimension D via log-log regression.
    Returns: (D, r_squared, box_sizes, counts)
    """
    max_box = min(gray.shape) // 4
    # Geometric sequence of box sizes
    sizes = []
    s = min_box
    while s <= max_box:
        sizes.append(int(s))
        s *= factor
    sizes = np.unique(sizes).astype(float)

    counts = box_count(gray, sizes)
    # Remove zeros before log
    mask = counts > 0
    log_s = np.log(sizes[mask])
    log_n = np.log(counts[mask])

    slope, intercept, r, p, se = stats.linregress(log_s, log_n)
    D = -slope
    return D, r**2, sizes, counts

In [ ]:
# Compute D per RGB channel
results = {}
for name, img, gray in [('Manet', img_m, gray_m), ('Contemporary', img_c, gray_c)]:
    D_rgb = []
    R2_rgb = []
    for ch, ch_name in enumerate(['R', 'G', 'B']):
        D, R2, sizes, counts = fractal_dimension(img[..., ch])
        D_rgb.append(D)
        R2_rgb.append(R2)
    D_gray, R2_gray, sizes, counts = fractal_dimension(gray)
    results[name] = {'D_R': D_rgb[0], 'D_G': D_rgb[1], 'D_B': D_rgb[2],
                     'D_gray': D_gray, 'R2': R2_gray,
                     'sizes': sizes, 'counts': counts}
    print(f"{name}: D_gray={D_gray:.4f}  R²={R2_gray:.4f}  "
          f"D_R={D_rgb[0]:.4f}  D_G={D_rgb[1]:.4f}  D_B={D_rgb[2]:.4f}")

In [ ]:
# ── Log-log plot: the key diagnostic ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, res) in zip(axes, results.items()):
    sizes, counts = res['sizes'], res['counts']
    mask = counts > 0
    log_s = np.log(sizes[mask])
    log_n = np.log(counts[mask])

    slope, intercept = np.polyfit(log_s, log_n, 1)
    ax.scatter(log_s, log_n, s=50, zorder=5, label='Data')
    ax.plot(log_s, slope * log_s + intercept, 'r--',
            label=f'Fit: D = {-slope:.4f}  R²={res["R2"]:.4f}')

    ax.set_xlabel('log(box size ε)')
    ax.set_ylabel('log(N(ε))')
    ax.set_title(f'{name} — log-log regression')
    ax.legend()
    ax.grid(alpha=0.3)

    # Annotate the linear region
    ax.annotate('Linear region\n(reliable D estimate)',
                xy=(log_s[len(log_s)//2], log_n[len(log_n)//2]),
                xytext=(log_s[1], log_n[-2]),
                arrowprops=dict(arrowstyle='->', color='grey'),
                color='grey', fontsize=9)

plt.suptitle('Box-counting: N(ε) vs ε  —  slope = −D', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Expected range:** $D \approx 1.3$–$1.7$ for oil paintings. Compare the two values — is there already a visible difference?

**Check:** Is the log-log relationship linear? If it curves strongly, the image may not be fractal at these scales — which is actually expected for representational paintings like Manet (less so than Pollock).

---
## 2. Sliding Window Fractal Dimension (SWFD) — Spatial D-Map

Instead of computing a single global $D$, we compute it **locally per patch** using a sliding window. The result is a spatial map $D(x,y)$ that shows which regions of the painting are more or less complex.

**Hypothesis for Manet:** Flat colour areas should show low $D$; contour edges and textured areas should show high $D$.

In [ ]:
def swfd_map(gray: np.ndarray, patch_size: int = 64, stride: int = 32) -> np.ndarray:
    """
    Compute a spatial map of local fractal dimensions.
    Returns array of shape (n_rows, n_cols) with D per patch.
    """
    H, W = gray.shape
    rows = range(0, H - patch_size + 1, stride)
    cols = range(0, W - patch_size + 1, stride)
    D_map = np.zeros((len(rows), len(cols)))

    for i, r in enumerate(rows):
        for j, c in enumerate(cols):
            patch = gray[r:r+patch_size, c:c+patch_size]
            # Need enough variation to estimate D
            if patch.std() < 1e-3:
                D_map[i, j] = np.nan
            else:
                D, *_ = fractal_dimension(patch, min_box=2)
                D_map[i, j] = D
    return D_map


print('Computing SWFD maps (may take ~30s per image)...')
dmap_m = swfd_map(gray_m, patch_size=64, stride=32)
dmap_c = swfd_map(gray_c, patch_size=64, stride=32)
print(f'D-map shapes: Manet={dmap_m.shape}  Contemporary={dmap_c.shape}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, (name, img, dmap) in enumerate([
    ('Manet', img_m, dmap_m),
    ('Contemporary', img_c, dmap_c)
]):
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'{name} — original', fontsize=12)
    axes[0, col].axis('off')

    vmin = np.nanpercentile(np.concatenate([dmap_m.ravel(), dmap_c.ravel()]), 5)
    vmax = np.nanpercentile(np.concatenate([dmap_m.ravel(), dmap_c.ravel()]), 95)

    im = axes[1, col].imshow(dmap, cmap='plasma', vmin=vmin, vmax=vmax,
                              interpolation='nearest')
    plt.colorbar(im, ax=axes[1, col], fraction=0.03, pad=0.02, label='D')
    axes[1, col].set_title(f'{name} — D-map (SWFD)', fontsize=12)
    axes[1, col].axis('off')

plt.suptitle('Sliding Window Fractal Dimension Map\n'
             'High D (yellow) = complex texture | Low D (purple) = flat/uniform area',
             fontsize=12)
plt.tight_layout()
plt.show()

# Distribution comparison
fig, ax = plt.subplots(figsize=(8, 4))
for name, dmap, color in [('Manet', dmap_m, '#2196F3'), ('Contemporary', dmap_c, '#FF5722')]:
    d_valid = dmap[~np.isnan(dmap)].ravel()
    ax.hist(d_valid, bins=40, density=True, alpha=0.6, color=color, label=f'{name}  μ={d_valid.mean():.3f}  σ={d_valid.std():.3f}')
ax.set_xlabel('Local fractal dimension D')
ax.set_ylabel('Density')
ax.set_title('Distribution of local D values')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Lacunarity λ(ε)

Lacunarity measures spatial **clustering and gappiness** — it captures how heterogeneously distributed the texture is, independent of D.

$$\lambda(\varepsilon) = \frac{\sigma^2(\varepsilon)}{\mu^2(\varepsilon)}$$

where $\sigma^2(\varepsilon)$ and $\mu^2(\varepsilon)$ are the variance and squared mean of pixel mass (total intensity) per box of side $\varepsilon$.

- **High lacunarity** = large gaps between dense regions (heterogeneous)
- **Low lacunarity** ≈ translationally uniform

Two paintings can have the same $D$ but very different $\lambda(\varepsilon)$ — lacunarity carries independent information.

In [ ]:
def lacunarity_curve(gray: np.ndarray, box_sizes: np.ndarray) -> np.ndarray:
    """
    Compute lacunarity λ(ε) = σ²/μ² for each box size.
    Uses pixel mass (sum of intensities) per box.
    """
    lacs = []
    H, W = gray.shape
    for s in box_sizes:
        s = int(s)
        h_trim = (H // s) * s
        w_trim = (W // s) * s
        blocks = gray[:h_trim, :w_trim].reshape(H // s, s, W // s, s)
        # Mass per box = sum of intensities
        mass = blocks.sum(axis=(1, 3)).ravel()
        mu = mass.mean()
        if mu < 1e-9:
            lacs.append(np.nan)
        else:
            lacs.append(mass.var() / (mu ** 2))
    return np.array(lacs)


# Box sizes: same geometric sequence used for D
box_sizes = np.unique([int(2 * 1.5**k) for k in range(20)
                       if 2 * 1.5**k <= min(gray_m.shape) // 4]).astype(float)

lac_m = lacunarity_curve(gray_m, box_sizes)
lac_c = lacunarity_curve(gray_c, box_sizes)

fig, ax = plt.subplots(figsize=(9, 5))
mask_m = ~np.isnan(lac_m)
mask_c = ~np.isnan(lac_c)

ax.plot(box_sizes[mask_m], lac_m[mask_m], 'o-', color='#2196F3', label='Manet')
ax.plot(box_sizes[mask_c], lac_c[mask_c], 's-', color='#FF5722', label='Contemporary')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Box size ε (pixels)')
ax.set_ylabel('Lacunarity λ(ε)')
ax.set_title('Lacunarity curves  —  λ(ε) = σ²(ε) / μ²(ε)')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print('Summary at intermediate scales:')
mid = len(box_sizes) // 2
print(f'  ε={box_sizes[mid]:.0f}px  Manet λ={lac_m[mid]:.4f}  Contemporary λ={lac_c[mid]:.4f}')

---
## 4. Generalised Dimension D(q) — Multifractal Spectrum

Instead of a single $D$, the full **multifractal spectrum** $D(q)$ captures the scaling properties of different statistical moments of the intensity distribution.

Define a probability measure in each box:
$$p_i(\varepsilon) = \frac{\text{mass in box } i}{\text{total mass}}$$

The partition function:
$$Z(q, \varepsilon) = \sum_i p_i^q \propto \varepsilon^{\tau(q)}, \qquad \tau(q) = (q-1) D(q)$$

So:
$$D(q) = \lim_{\varepsilon \to 0} \frac{1}{q-1} \frac{\log Z(q,\varepsilon)}{\log \varepsilon}$$

Special cases:
- $D(0)$ = box-counting dimension (what we computed above)
- $D(1)$ = information dimension (Shannon entropy of the measure)
- $D(2)$ = correlation dimension

For a **monofractal**, $D(q) = \text{const}$. For a **multifractal**, $D(q)$ decreases with $q$ — the curve carries discriminative information.

In [ ]:
def generalised_dimension(gray: np.ndarray, box_sizes: np.ndarray,
                          q_values: np.ndarray) -> np.ndarray:
    """
    Estimate D(q) for a range of q values via log-log regression of Z(q,ε).
    Returns D(q) array of same length as q_values.
    Handles q=1 (information dimension) via L'Hôpital / entropy limit.
    """
    H, W = gray.shape
    total_mass = gray.sum()
    if total_mass < 1e-9:
        return np.full_like(q_values, np.nan)

    D_q = []
    for q in q_values:
        log_Z = []
        log_eps = []
        for s in box_sizes:
            s = int(s)
            h_trim = (H // s) * s
            w_trim = (W // s) * s
            blocks = gray[:h_trim, :w_trim].reshape(H // s, s, W // s, s)
            mass = blocks.sum(axis=(1, 3)).ravel()
            p = mass / (mass.sum() + 1e-12)  # normalise to probability
            p = p[p > 0]

            if abs(q - 1.0) < 1e-6:  # q→1: entropy dimension
                Z = -np.sum(p * np.log(p + 1e-300))
            else:
                Z = np.sum(p ** q)

            if Z > 0:
                log_Z.append(np.log(Z))
                log_eps.append(np.log(s))

        if len(log_Z) < 3:
            D_q.append(np.nan)
            continue

        slope, *_ = np.polyfit(log_eps, log_Z, 1)
        if abs(q - 1.0) < 1e-6:
            D_q.append(-slope)   # entropy dimension
        else:
            D_q.append(slope / (q - 1))

    return np.array(D_q)


q_values = np.linspace(-5, 5, 41)   # q from -5 to +5
print('Computing D(q) spectra (takes ~1 min)...')
Dq_m = generalised_dimension(gray_m, box_sizes, q_values)
Dq_c = generalised_dimension(gray_c, box_sizes, q_values)
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(q_values, Dq_m, 'o-', color='#2196F3', ms=4, label='Manet')
ax.plot(q_values, Dq_c, 's-', color='#FF5722', ms=4, label='Contemporary')

# Reference lines
ax.axvline(0, color='grey', lw=0.8, ls='--', alpha=0.5)
ax.axvline(1, color='grey', lw=0.8, ls='--', alpha=0.5)
ax.axvline(2, color='grey', lw=0.8, ls='--', alpha=0.5)
ax.text(0.05, ax.get_ylim()[0]+0.02, 'D(0)\nbox-counting', fontsize=8, color='grey', transform=ax.get_xaxis_transform())
ax.text(1.05, ax.get_ylim()[0]+0.02, 'D(1)\nentropy', fontsize=8, color='grey', transform=ax.get_xaxis_transform())
ax.text(2.05, ax.get_ylim()[0]+0.02, 'D(2)\ncorrelation', fontsize=8, color='grey', transform=ax.get_xaxis_transform())

ax.set_xlabel('q')
ax.set_ylabel('D(q)')
ax.set_title('Generalised dimension spectrum D(q)\nA flat curve = monofractal; decreasing = multifractal')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Manet:        D(0)={Dq_m[q_values==0.0].item():.4f}  D(1)≈{Dq_m[np.argmin(np.abs(q_values-1))].item():.4f}  D(2)={Dq_m[np.argmin(np.abs(q_values-2))].item():.4f}")
print(f"Contemporary: D(0)={Dq_c[q_values==0.0].item():.4f}  D(1)≈{Dq_c[np.argmin(np.abs(q_values-1))].item():.4f}  D(2)={Dq_c[np.argmin(np.abs(q_values-2))].item():.4f}")

---
## 5. Singularity Spectrum f(α) — Legendre Transform

The singularity spectrum $f(\alpha)$ is the Legendre transform of the scaling exponents $\tau(q) = (q-1)D(q)$:

$$\alpha(q) = \frac{d\tau(q)}{dq}, \qquad f(\alpha) = q\,\alpha(q) - \tau(q)$$

Geometrically:
- $\alpha$ is the **Hölder exponent** — the local singularity strength
- $f(\alpha)$ is the **fractal dimension** of the set of points with that Hölder exponent
- The width $\Delta\alpha = \alpha_{\max} - \alpha_{\min}$ is the key discriminative feature: larger $\Delta\alpha$ = stronger multifractality

A **monofractal** gives a single point $f = D$. A **multifractal** gives an inverted parabola.

In [ ]:
def singularity_spectrum(q_values: np.ndarray, Dq: np.ndarray) -> tuple:
    """
    Compute f(α) via Legendre transform of τ(q) = (q-1)·D(q).
    Returns (alpha, f_alpha, delta_alpha).
    """
    # Mask NaN
    mask = ~np.isnan(Dq)
    q = q_values[mask]
    Dq_clean = Dq[mask]

    tau = (q - 1) * Dq_clean                    # τ(q) = (q-1)D(q)
    # Numerical derivative dτ/dq
    alpha = np.gradient(tau, q)                  # α(q) = dτ/dq
    f_alpha = q * alpha - tau                    # f(α) = q·α − τ

    delta_alpha = alpha.max() - alpha.min()
    return alpha, f_alpha, delta_alpha


alpha_m, f_m, Dalpha_m = singularity_spectrum(q_values, Dq_m)
alpha_c, f_c, Dalpha_c = singularity_spectrum(q_values, Dq_c)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(alpha_m, f_m, 'o-', color='#2196F3', ms=4,
        label=f'Manet  Δα={Dalpha_m:.4f}')
ax.plot(alpha_c, f_c, 's-', color='#FF5722', ms=4,
        label=f'Contemporary  Δα={Dalpha_c:.4f}')

ax.set_xlabel('α  (Hölder exponent)')
ax.set_ylabel('f(α)  (fractal dimension of iso-α set)')
ax.set_title('Singularity spectrum f(α)\n'
             'Δα = spectrum width — key discriminative feature\n'
             'Narrow Δα = near-monofractal | Wide = strong multifractal')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Manet:        Δα = {Dalpha_m:.4f}  (α range: [{alpha_m.min():.3f}, {alpha_m.max():.3f}])")
print(f"Contemporary: Δα = {Dalpha_c:.4f}  (α range: [{alpha_c.min():.3f}, {alpha_c.max():.3f}])")

---
## 6. Summary — Feature Vector

The full feature vector extracted from one image for the SWFD analyzer:

In [ ]:
def extract_fractal_features(gray: np.ndarray, img_rgb: np.ndarray) -> dict:
    """Extract the full fractal feature vector for one image."""
    box_sizes_local = np.unique([int(2 * 1.5**k) for k in range(20)
                                 if 2 * 1.5**k <= min(gray.shape) // 4]).astype(float)
    q_vals = np.linspace(-5, 5, 21)

    # Per-channel D
    D_per_ch = {}
    for ch, name in enumerate(['R', 'G', 'B']):
        D, R2, *_ = fractal_dimension(img_rgb[..., ch])
        D_per_ch[f'D_{name}'] = D
        D_per_ch[f'R2_{name}'] = R2

    # D-map statistics
    dmap = swfd_map(gray)
    d_valid = dmap[~np.isnan(dmap)].ravel()

    # Multifractal
    Dq = generalised_dimension(gray, box_sizes_local, q_vals)
    alpha, f_alpha, delta_alpha = singularity_spectrum(q_vals, Dq)

    # Lacunarity at 5 scales
    lac = lacunarity_curve(gray, box_sizes_local)
    lac_sample = lac[~np.isnan(lac)][::max(1, len(lac)//5)][:5]

    features = {
        **D_per_ch,
        'D_map_mean': d_valid.mean(),
        'D_map_std':  d_valid.std(),
        'D_map_skew': float(np.mean(((d_valid - d_valid.mean()) / (d_valid.std() + 1e-9))**3)),
        'D_map_kurt': float(np.mean(((d_valid - d_valid.mean()) / (d_valid.std() + 1e-9))**4)),
        'delta_alpha': delta_alpha,
        'alpha_max':  alpha.max(),
        'alpha_min':  alpha.min(),
        **{f'lac_{i}': v for i, v in enumerate(lac_sample)},
    }
    return features


print('Extracting full feature vectors...')
feat_m = extract_fractal_features(gray_m, img_m)
feat_c = extract_fractal_features(gray_c, img_c)

import pandas as pd
df = pd.DataFrame({'Manet': feat_m, 'Contemporary': feat_c})
df['Difference'] = (df['Manet'] - df['Contemporary']).abs()
print('\nFeature comparison:')
print(df.round(4).to_string())

---
## 7. Key Observations

Fill in after running the notebook:

| Question | Observation |
|---|---|
| Is the log-log plot linear? | |
| D-map: do flat colour areas show low D? | |
| Lacunarity curves: do they separate? | |
| D(q) curve: monofractal or multifractal? | |
| Δα: Manet vs contemporary? | |
| Overall: discriminative enough for Tier 1? | |

**Decision:** Based on visual separability, decide whether fractal methods stay in Tier 1 or move to Tier 2 for Manet specifically.